# Shorts Tok Swarm - Video Generator

**Before running:** Runtime -> Change runtime type -> T4 GPU

## Steps
1. Run **Cell 1 only** - runtime will restart automatically
2. After restart, **skip Cell 1** and run from Cell 2 onwards

## Workflow
- Upload your local  folder to Google Drive at:
  
- Set  in Cell 3
- Final video saved back to Drive at 


In [ ]:
# Cell 1: Install dependencies + restart runtime
# After this cell finishes, runtime will restart automatically.
# Then run from Cell 2 onwards (skip Cell 1).
!pip install diffusers transformers accelerate sentencepiece -q
!apt-get install -y ffmpeg libass-dev > /dev/null 2>&1
print("Packages installed. Restarting runtime...")
import os
os.kill(os.getpid(), 9)  # Force restart to clear torch conflicts


In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Config — set your JOB_ID here
import os
import shutil

JOB_ID = "20260404_131120"  # <-- change this to your job folder name

DRIVE_JOB_PATH = f'/content/drive/MyDrive/ShortsTokSwarm/jobs/{JOB_ID}'
WORK_DIR = f'/content/job_{JOB_ID}'

os.makedirs(WORK_DIR, exist_ok=True)

files_needed = ['script.txt', 'audio.mp3', 'captions.srt', 'scene_prompts.json']
for fname in files_needed:
    src = os.path.join(DRIVE_JOB_PATH, fname)
    dst = os.path.join(WORK_DIR, fname)
    shutil.copy2(src, dst)
    print(f'  Copied: {fname}')

print(f'Job files ready at {WORK_DIR}')
print(os.listdir(WORK_DIR))


In [ ]:
# Cell 4: Load job data
import json

with open(f"{WORK_DIR}/scene_prompts.json") as f:
    scenes = json.load(f)

with open(f"{WORK_DIR}/script.txt") as f:
    script = f.read()

AUDIO_PATH = f"{WORK_DIR}/audio.mp3"
CAPTIONS_PATH = f"{WORK_DIR}/captions.srt"

print(f"{len(scenes)} scenes loaded")
for i, s in enumerate(scenes):
    print(f"  [{i+1}] {s[:80]}...")

In [ ]:
# Cell 5: Load LTX-Video model
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()  # Manages VRAM automatically
pipe.enable_vae_slicing()        # Reduces peak VRAM usage

print("Model loaded.")

In [ ]:
# Cell 6: Generate video clips
# Each clip: 9:16, 97 frames (~4s at 24fps)
# Then slowed 2.5x in assembly = ~10s per clip

import gc

NEGATIVE_PROMPT = (
    "blurry, low quality, distorted faces, text overlay, watermark, "
    "cartoon, anime, drawing, oversaturated, motion blur, duplicate"
)

clip_paths = []

for i, scene_prompt in enumerate(scenes):
    print(f"Generating clip {i+1}/{len(scenes)}...")
    print(f"  Prompt: {scene_prompt[:80]}")

    try:
        result = pipe(
            prompt=scene_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            width=432,
            height=768,
            num_frames=97,          # ~4 seconds, safe for T4
            num_inference_steps=40,
            guidance_scale=3.0,
        )

        clip_path = f"{WORK_DIR}/clip_{i:03d}_raw.mp4"
        export_to_video(result.frames[0], clip_path, fps=24)
        clip_paths.append(clip_path)
        print(f"  Saved: {clip_path}")

    except torch.cuda.OutOfMemoryError:
        print(f"  OOM on clip {i+1}. Retrying at lower frame count (65 frames)...")
        torch.cuda.empty_cache()
        gc.collect()

        result = pipe(
            prompt=scene_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            width=432,
            height=768,
            num_frames=65,
            num_inference_steps=40,
            guidance_scale=3.0,
        )
        clip_path = f"{WORK_DIR}/clip_{i:03d}_raw.mp4"
        export_to_video(result.frames[0], clip_path, fps=24)
        clip_paths.append(clip_path)
        print(f"  Saved (reduced): {clip_path}")

    torch.cuda.empty_cache()
    gc.collect()

print(f"\nAll {len(clip_paths)} clips generated.")

In [ ]:
# Cell 7: Slow down each clip to ~10s, upscale to 1080x1920
import subprocess

processed_paths = []

for i, raw_path in enumerate(clip_paths):
    out_path = f"{WORK_DIR}/clip_{i:03d}.mp4"

    # setpts=2.5*PTS slows clip to ~10s, scale to 1080x1920
    cmd = [
        "ffmpeg", "-y", "-i", raw_path,
        "-vf", "setpts=2.5*PTS,scale=1080:1920:flags=lanczos",
        "-r", "24",
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "20",
        out_path
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    processed_paths.append(out_path)
    print(f"Processed clip {i+1}/{len(clip_paths)}")

print("All clips processed.")

In [ ]:
# Cell 8: Concatenate clips + merge audio + burn captions
import subprocess

# Write concat list
concat_file = f"{WORK_DIR}/concat.txt"
with open(concat_file, "w") as f:
    for p in processed_paths:
        f.write(f"file '{p}'\n")

# Step 1: Concatenate clips (no audio yet)
concat_video = f"{WORK_DIR}/concat.mp4"
subprocess.run([
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0", "-i", concat_file,
    "-c", "copy",
    concat_video
], check=True, capture_output=True)
print("Clips concatenated.")

# Step 2: Add audio + burn captions
final_path = f"{WORK_DIR}/final.mp4"
caption_style = (
    "FontName=Arial,FontSize=22,PrimaryColour=&Hffffff,"
    "OutlineColour=&H000000,Outline=2,Bold=1,"
    "Alignment=2,MarginV=60"
)
subprocess.run([
    "ffmpeg", "-y",
    "-i", concat_video,
    "-i", AUDIO_PATH,
    "-vf", f"subtitles={CAPTIONS_PATH}:force_style='{caption_style}'",
    "-c:v", "libx264", "-preset", "fast", "-crf", "20",
    "-c:a", "aac", "-b:a", "192k",
    "-shortest",
    final_path
], check=True, capture_output=True)
print(f"Final video: {final_path}")

# Get duration info
result = subprocess.run(
    ["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", final_path],
    capture_output=True, text=True
)
duration = float(result.stdout.strip())
print(f"Duration: {duration:.1f}s")

if duration > 60:
    print("WARNING: Video is over 60s. Trimming for Shorts compatibility...")
    trimmed_path = f"{WORK_DIR}/final_trimmed.mp4"
    subprocess.run([
        "ffmpeg", "-y", "-i", final_path,
        "-t", "60", "-c", "copy", trimmed_path
    ], check=True, capture_output=True)
    final_path = trimmed_path
    print(f"Trimmed to 60s: {trimmed_path}")

In [ ]:
# Cell 9: Save final video back to Google Drive
import shutil

drive_output = f"{DRIVE_JOB_PATH}/final.mp4"
shutil.copy(final_path, drive_output)
print(f"Saved to Drive: {drive_output}")
print()
print("Download it from Google Drive, then run the publisher to upload to platforms.")

In [ ]:
# Cell 10: Preview (optional)
from IPython.display import Video
Video(final_path, width=360)